# Visualize Sphere Latent Space

This notebook visualizes the latent space of a trained Sphere Encoder model. The latents are normalized vectors on a unit sphere.

In [40]:
import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from torchvision import datasets, transforms
from sphere.model import G
from sphere.utils import load_ckpt
from sphere.loader import create_dataset

In [41]:
# Configuration
job_dir = "sphere-small-small-cifar-10-32px"  # Change this to your experiment name
dev_dir = "workspace"
num_samples = 5000  # Number of samples to visualize
use_pca = True  # Use PCA to reduce to 3D, otherwise use first 3 dimensions
use_tsne = False  # Use t-SNE instead of PCA (slower but potentially better)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [42]:
# Load model configuration
exp_dir = os.path.join(dev_dir, "experiments", job_dir)
config_path = os.path.join(exp_dir, "cfg.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

print("Model configuration:")
for k, v in cfg.items():
    print(f"{k}: {v}")

Model configuration:
dev_dir: workspace
out_dir: experiments
data_dir: /scratch-local/scur0199/datasets/cifar-10
log_interval: 100
vis_interval: 2
ckpt_save_interval: 100
early_stop_patience: 15
class_of_interest: None
forward_steps: 2
cfg_position: combo
cfg: 1.0
use_wandb: False
wandb_project: None
wandb_entity: None
wandb_key: None
dataset_name: cifar-10
image_size: 32
num_workers: 12
crop_mode: center
flip_image: True
extra_padding: False
rot_degrees: 0
interp_mode: nearest
concat_train_val_splits: False
load_from_zip: False
max_samples: -1
batch_size: 256
batch_size_per_rank: 128
warmup_epochs: 10
weight_decay: 0.0
grad_clip: 1.0
epochs: 1000
learning_rate: 0.0001
min_lr: 1e-06
encoder_lr_scaler: 0.1
decay_lr: True
compression_ratio: 3.0
latent_resolution: 16
noise_sigma_max_angle: 80
mix_hard_cases: True
mix_hard_cases_prob: 0.1
vit_enc_model_size: small
vit_dec_model_size: small
vit_enc_latent_mlp_mixer_depth: 2
vit_dec_latent_mlp_mixer_depth: 2
affine_latent_mlp_mixer: True
con

In [43]:
# Build model
model = G(
    input_size=cfg['image_size'],
    patch_size=cfg['patch_size'],
    vit_enc_model_size=cfg['vit_enc_model_size'],
    vit_dec_model_size=cfg['vit_dec_model_size'],
    token_channels=cfg['token_channels'],
    num_classes=cfg['num_classes'] if cfg.get('cond_generator', False) else 0,
    halve_model_size=cfg.get('halve_model_size', False),
    spherify_model=cfg.get('spherify_model', False),
    pixel_head_type=cfg.get('pixel_head_type', 'linear'),
    in_context_size=cfg.get('in_context_size', 0),
    noise_sigma_max_angle=cfg.get('noise_sigma_max_angle', 85.0),
    vit_enc_latent_mlp_mixer_depth=cfg.get('vit_enc_latent_mlp_mixer_depth', 0),
    vit_dec_latent_mlp_mixer_depth=cfg.get('vit_dec_latent_mlp_mixer_depth', 0),
    affine_latent_mlp_mixer=cfg.get('affine_latent_mlp_mixer', True),
)

# Load checkpoint
ckpt_dir = os.path.join(exp_dir, "ckpt")
ckpts = sorted(os.listdir(ckpt_dir))
latest_ckpt = os.path.join(ckpt_dir, ckpts[-1])  # Load latest checkpoint
print(f"Loading checkpoint: {latest_ckpt}")

load_ckpt(model, ckpt_path=latest_ckpt, strict=True)
model.eval()
print("Model loaded successfully")

Loading checkpoint: workspace/experiments/sphere-small-small-cifar-10-32px/ckpt/ep0172.pth


/home/sakr/UvA/DL 2/spherical-flow-matching/sphere-encoder-main/sphere/utils.py:303: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=

Model loaded successfully


In [ ]:
model = model.to(device)

In [45]:
# Load dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

if cfg['dataset_name'] in ['cifar-10']:
    dataset_cls = datasets.__dict__[cfg['dataset_name'].upper().replace('-', '')]
    dataset = create_dataset(
        dataset_cls,
        root=os.path.join(dev_dir, "datasets", cfg['dataset_name']),
        split="train",
        download=True,
        transform=transform,
    )
else:
    dataset = datasets.ImageFolder(
        root=os.path.join(dev_dir, "datasets", cfg['dataset_name']),
        transform=transform
    )

# Subset for visualization
indices = np.random.choice(len(dataset), num_samples, replace=False)
subset = torch.utils.data.Subset(dataset, indices)
dataloader = torch.utils.data.DataLoader(subset, batch_size=32, shuffle=False)

print(f"Loaded {len(subset)} samples from {cfg['dataset_name']}")

Files already downloaded and verified
Loaded 5000 samples from cifar-10


In [ ]:
# Extract latents
latents = []
labels = []

with torch.no_grad():
    for images, targets in dataloader:
        images = images.to(device)
        targets = targets.to(device)
        z = model.encoder(images, y=targets)
        z_avg = z.mean(dim=1)
        latents.append(z_avg.cpu().numpy())
        labels.append(targets.cpu().numpy())

latents = np.concatenate(latents, axis=0)
labels = np.concatenate(labels, axis=0)

print(f"Latents shape: {latents.shape}")
print(f"Labels shape: {labels.shape}")

Latents shape: (5000, 4)
Labels shape: (5000,)


In [ ]:
latents_norm = latents / np.linalg.norm(latents, axis=-1, keepdims=True)
print("Latents normalized to unit sphere")

Latents normalized to unit sphere


In [ ]:
if latents_norm.shape[-1] > 3:
    if use_tsne:
        print("Using t-SNE for dimensionality reduction...")
        tsne = TSNE(n_components=3, random_state=42)
        latents_3d = tsne.fit_transform(latents_norm.reshape(latents_norm.shape[0], -1))
    elif use_pca:
        print("Using PCA for dimensionality reduction...")
        pca = PCA(n_components=3)
        latents_3d = pca.fit_transform(latents_norm.reshape(latents_norm.shape[0], -1))
    else:
        print("Using first 3 dimensions...")
        latents_3d = latents_norm.reshape(latents_norm.shape[0], -1)[:, :3]
else:
    latents_3d = latents_norm.reshape(latents_norm.shape[0], -1)

latents_3d = latents_3d / np.linalg.norm(latents_3d, axis=1, keepdims=True)

print(f"Reduced latents shape: {latents_3d.shape}")

Using PCA for dimensionality reduction...
Reduced latents shape: (5000, 3)


In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
class_labels_text = [class_names[int(label)] for label in labels]

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=latents_3d[:, 0],
    y=latents_3d[:, 1],
    z=latents_3d[:, 2],
    mode='markers',
    marker=dict(
        size=4,
        color=labels,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Class', tickvals=list(range(10)), ticktext=class_names),
        opacity=0.7
    ),
    text=class_labels_text,
    hovertemplate='<b>%{text}</b><br>X: %{x:.3f}<br>Y: %{y:.3f}<br>Z: %{z:.3f}<extra></extra>',
    name='Samples'
))

u = np.linspace(0, 2 * np.pi, 50)
v = np.linspace(0, np.pi, 50)
x_sphere = np.outer(np.cos(u), np.sin(v))
y_sphere = np.outer(np.sin(u), np.sin(v))
z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))

fig.add_trace(go.Surface(
    x=x_sphere,
    y=y_sphere,
    z=z_sphere,
    opacity=0.15,
    colorscale='Blues',
    showscale=False,
    name='Unit Sphere',
    hoverinfo='skip'
))

fig.update_layout(
    title=f'Interactive Latent Space Visualization - {job_dir}',
    scene=dict(
        xaxis=dict(range=[-1.2, 1.2]),
        yaxis=dict(range=[-1.2, 1.2]),
        zaxis=dict(range=[-1.2, 1.2]),
    ),
    width=1000,
    height=800,
    hovermode='closest'
)

fig.show()

In [ ]:
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans

silhouette_avg = silhouette_score(latents_3d, labels)
print(f"Silhouette Score (higher is better separation): {silhouette_avg:.3f}")

kmeans = KMeans(n_clusters=len(np.unique(labels)), random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(latents_3d)
adjusted_rand_score_val = adjusted_rand_score(labels, cluster_labels)
print(f"Adjusted Rand Index (1.0 = perfect clustering): {adjusted_rand_score_val:.3f}")

class_centroids = []
for cls in np.unique(labels):
    mask = labels == cls
    centroid = latents_3d[mask].mean(axis=0)
    class_centroids.append(centroid)

class_centroids = np.array(class_centroids)

fig = go.Figure()

u = np.linspace(0, 2 * np.pi, 50)
v = np.linspace(0, np.pi, 50)
x_sphere = np.outer(np.cos(u), np.sin(v))
y_sphere = np.outer(np.sin(u), np.sin(v))
z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))

fig.add_trace(go.Surface(
    x=x_sphere,
    y=y_sphere,
    z=z_sphere,
    opacity=0.15,
    colorscale='Blues',
    showscale=False,
    name='Unit Sphere',
    hoverinfo='skip'
))

# Add all points
fig.add_trace(go.Scatter3d(
    x=latents_3d[:, 0],
    y=latents_3d[:, 1],
    z=latents_3d[:, 2],
    mode='markers',
    marker=dict(
        size=3,
        color=labels,
        colorscale='Viridis',
        showscale=False,
        opacity=0.4
    ),
    text=[class_names[int(label)] for label in labels],
    hovertemplate='<b>%{text}</b><br>X: %{x:.3f}<br>Y: %{y:.3f}<br>Z: %{z:.3f}<extra></extra>',
    name='Samples'
))

centroid_labels = [class_names[i] for i in range(len(class_centroids))]
fig.add_trace(go.Scatter3d(
    x=class_centroids[:, 0],
    y=class_centroids[:, 1],
    z=class_centroids[:, 2],
    mode='markers+text',
    marker=dict(
        size=10,
        color=list(range(len(class_centroids))),
        colorscale='Viridis',
        symbol='diamond',
        line=dict(color='black', width=2),
        opacity=0.9
    ),
    text=centroid_labels,
    textposition='top center',
    hovertemplate='<b>Centroid: %{text}</b><br>X: %{x:.3f}<br>Y: %{y:.3f}<br>Z: %{z:.3f}<extra></extra>',
    name='Class Centroids',
    showlegend=True
))

fig.update_layout(
    title=f'Latent Space with Class Centroids - {job_dir}',
    scene=dict(
        xaxis=dict(range=[-1.2, 1.2]),
        yaxis=dict(range=[-1.2, 1.2]),
        zaxis=dict(range=[-1.2, 1.2]),
    ),
    width=1000,
    height=800,
    hovermode='closest'
)

fig.show()
print("Class centroids plotted (marked with diamond symbols)")

Silhouette Score (higher is better separation): -0.099
Adjusted Rand Index (1.0 = perfect clustering): 0.028


Class centroids plotted (marked with diamond symbols)


In [ ]:
num_classes = len(np.unique(labels))
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

fig_classes = make_subplots(
    rows=2, cols=5,
    specs=[[{'type': 'scatter3d'} for _ in range(5)] for _ in range(2)],
    subplot_titles=[f'{class_names[cls]} (n={np.sum(labels == cls)})' for cls in range(num_classes)]
)

for cls in range(num_classes):
    mask = labels == cls
    class_points = latents_3d[mask]
    centroid = class_centroids[cls]
    
    row = (cls // 5) + 1
    col = (cls % 5) + 1
    
    u = np.linspace(0, 2 * np.pi, 30)
    v = np.linspace(0, np.pi, 30)
    x_sphere = np.outer(np.cos(u), np.sin(v))
    y_sphere = np.outer(np.sin(u), np.sin(v))
    z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))
    
    fig_classes.add_trace(
        go.Surface(
            x=x_sphere, y=y_sphere, z=z_sphere,
            opacity=0.1, colorscale='Blues', showscale=False,
            name='', hoverinfo='skip'
        ),
        row=row, col=col
    )
    
    fig_classes.add_trace(
        go.Scatter3d(
            x=class_points[:, 0],
            y=class_points[:, 1],
            z=class_points[:, 2],
            mode='markers',
            marker=dict(size=4, color=cls, colorscale='Viridis', opacity=0.8),
            name=class_names[cls],
            hovertemplate='<b>' + class_names[cls] + '</b><br>X: %{x:.3f}<br>Y: %{y:.3f}<br>Z: %{z:.3f}<extra></extra>',
            showlegend=False
        ),
        row=row, col=col
    )
    
    fig_classes.add_trace(
        go.Scatter3d(
            x=[centroid[0]],
            y=[centroid[1]],
            z=[centroid[2]],
            mode='markers',
            marker=dict(size=4, color='red', symbol='diamond', line=dict(color='black', width=2)),
            name='',
            hovertemplate='<b>Centroid</b><br>X: %{x:.3f}<br>Y: %{y:.3f}<br>Z: %{z:.3f}<extra></extra>',
            showlegend=False
        ),
        row=row, col=col
    )
    
    fig_classes.update_scenes(
        xaxis=dict(range=[-1.2, 1.2]),
        yaxis=dict(range=[-1.2, 1.2]),
        zaxis=dict(range=[-1.2, 1.2]),
        row=row, col=col
    )

fig_classes.update_layout(
    title_text=f'Interactive Per-Class Latent Space - {job_dir}',
    height=900,
    width=1600,
    showlegend=False
)

fig_classes.show()

In [ ]:
print("Per-class statistics:")
print("-" * 80)
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

for cls in range(len(np.unique(labels))):
    mask = labels == cls
    class_points = latents_3d[mask]

    num_samples = len(class_points)
    centroid = class_centroids[cls]
    distances_to_centroid = np.linalg.norm(class_points - centroid, axis=1)
    mean_distance = distances_to_centroid.mean()
    std_distance = distances_to_centroid.std()

    print(f"Class {cls} ({class_names[cls]:>10}): {num_samples:3d} samples, "
          f"mean dist: {mean_distance:.3f}, std dist: {std_distance:.3f}")

print("-" * 80)

Per-class statistics:
--------------------------------------------------------------------------------
Class 0 (  airplane): 483 samples, mean dist: 0.859, std dist: 0.246
Class 1 (automobile): 497 samples, mean dist: 0.948, std dist: 0.145
Class 2 (      bird): 504 samples, mean dist: 0.978, std dist: 0.107
Class 3 (       cat): 496 samples, mean dist: 0.960, std dist: 0.129
Class 4 (      deer): 505 samples, mean dist: 0.954, std dist: 0.157
Class 5 (       dog): 511 samples, mean dist: 0.976, std dist: 0.108
Class 6 (      frog): 504 samples, mean dist: 0.930, std dist: 0.168
Class 7 (     horse): 489 samples, mean dist: 0.979, std dist: 0.096
Class 8 (      ship): 508 samples, mean dist: 0.840, std dist: 0.236
Class 9 (     truck): 503 samples, mean dist: 0.939, std dist: 0.154
--------------------------------------------------------------------------------
